# 2. Learn N-gram / Markov probabilities and generate a sentence

This notebook reads the `observations.csv` generated by Notebook 1 and:

1. computes N-grams for a configurable `N`;
2. estimates Markov / N-gram conditional probabilities from the observations;
3. starts from a configurable `START` sequence;
4. generates the continuation **one token at a time by sampling from the learned probabilities**.

Run this notebook after running `01_generate_observations.ipynb`.

### Important idea

For an N-gram model, the Markov assumption is:

\[
P(w_t \mid w_1,\ldots,w_{t-1})
\approx
P(w_t \mid w_{t-N+1},\ldots,w_{t-1}).
\]

Thus the state is the previous `N-1` tokens.


In [1]:
import random
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_FILE = "observations.csv"

# N=2 -> bigram model
# N=3 -> trigram model
# N=4 -> four-gram model, etc.
N = 2

# The initial sentence/prefix used by the generator.
# Examples:
#   START = "s"
#   START = "s r"
#   START = "s l l"
#   START = "i l"
START = "s"

# Set to None for different output on every execution.
RANDOM_SEED = None

# Maximum number of generated tokens after START.
MAX_GENERATED_TOKENS = 20

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)


In [2]:
# ------------------------------------------------------------
# Read observations
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"Read {len(df)} observations.")
df.head()


Read 850 observations.


,id,sentence
0,1,s f e
1,2,s l l f e
2,3,s f e
3,4,s r r f e
4,5,s f e


## Add sentence boundary markers

We add `<s>` and `</s>` so that the model can learn:

- how sentences begin;
- how sentences end.

For example:

```text
<s> s r r f e </s>
```

This is especially useful because generation can then stop when the model samples `</s>`.


In [3]:
def tokenize(sentence):
    return sentence.split()

sequences = [
    ["<s>"] + tokenize(sentence) + ["</s>"]
    for sentence in df["sentence"]
]

sequences[:3]


[['<s>', 's', 'f', 'e', '</s>'],
 ['<s>', 's', 'l', 'l', 'f', 'e', '</s>'],
 ['<s>', 's', 'f', 'e', '</s>']]

## Compute N-grams

For an N-gram model, an N-gram contains `N` consecutive tokens.

For example, with `N = 3`:

```text
<s> s r
s r r
r r f
r f e
f e </s>
```

The corresponding Markov state is the first `N-1` tokens, and the next token is the predicted/emitted token.


In [4]:
def extract_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

all_ngrams = []

for sequence in sequences:
    all_ngrams.extend(extract_ngrams(sequence, N))

ngram_counts = Counter(all_ngrams)

ngram_table = (
    pd.DataFrame(
        [
            {"ngram": " ".join(ngram), "count": count}
            for ngram, count in ngram_counts.items()
        ]
    )
    .sort_values(["count", "ngram"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Number of distinct {N}-grams: {len(ngram_table)}")
ngram_table


Number of distinct 2-grams: 21


,ngram,count
0,r r,785
1,<s> s,763
2,e </s>,763
3,f e,546
4,l l,493
5,r f,325
6,s l,306
7,s r,304
8,l r,151
9,p e,145


## Estimate Markov probabilities

For an N-gram model:

$$
P(w_t \mid w_{t-N+1},\ldots,w_{t-1})
=
\frac{
C(w_{t-N+1},\ldots,w_t)
}{
C(w_{t-N+1},\ldots,w_{t-1})
}.
$$

In other words:

> **Markov probability = count of the complete N-gram / count of its `(N-1)`-token history.**

For a bigram model this becomes:

$$
P(w_t\mid w_{t-1})
=
\frac{C(w_{t-1},w_t)}
{C(w_{t-1})}.
$$


In [5]:
# Count histories (the first N-1 tokens of every N-gram)
history_counts = Counter()

for ngram, count in ngram_counts.items():
    history = ngram[:-1]
    history_counts[history] += count

# Conditional probabilities
markov = []

for ngram, count in ngram_counts.items():
    history = ngram[:-1]
    next_token = ngram[-1]

    probability = count / history_counts[history]

    markov.append({
        "history": " ".join(history),
        "next_token": next_token,
        "ngram_count": count,
        "history_count": history_counts[history],
        "probability": probability,
    })

markov_df = (
    pd.DataFrame(markov)
    .sort_values(["history", "probability", "next_token"],
                 ascending=[True, False, True])
    .reset_index(drop=True)
)

markov_df


,history,next_token,ngram_count,history_count,probability
0,<s>,s,763,850,0.897647
1,<s>,i,87,850,0.102353
2,e,</s>,763,763,1.000000
3,f,e,546,546,1.000000
4,i,l,87,87,1.000000
5,l,l,493,886,0.556433
6,l,r,151,886,0.170429
7,l,f,114,886,0.128668
8,l,</s>,87,886,0.098194
9,l,p,31,886,0.034989


In [6]:
# A more compact display of the learned transition probabilities.
# This is the Markov-chain view: state -> next token.

for history, group in markov_df.groupby("history", sort=True):
    transitions = [
        f"{row.next_token}: {row.probability:.3f}"
        for row in group.itertuples()
    ]
    print(f"{history:15s} -> " + ", ".join(transitions))


<s>             -> s: 0.898, i: 0.102
e               -> </s>: 1.000
f               -> e: 1.000
i               -> l: 1.000
l               -> l: 0.556, r: 0.170, f: 0.129, </s>: 0.098, p: 0.035, e: 0.011
p               -> e: 1.000
r               -> r: 0.633, f: 0.262, p: 0.067, e: 0.038
s               -> l: 0.401, r: 0.398, f: 0.140, p: 0.041, e: 0.020


## Generate a sentence step by step

At each step:

1. identify the current `(N-1)`-token state;
2. find all observed next-token transitions from that state;
3. sample one according to the learned probabilities;
4. append the selected token;
5. repeat until `</s>` is produced.

This is the practical Markov-chain interpretation of the N-gram language model.


In [7]:
# Build a transition table convenient for sampling.
transitions = defaultdict(list)

for row in markov_df.itertuples():
    history = tuple(row.history.split())
    transitions[history].append(
        (row.next_token, row.probability)
    )

def sample_next(history):
    """Sample the next token using learned Markov probabilities."""
    options = transitions.get(history)

    if not options:
        return None

    tokens = [token for token, _ in options]
    probabilities = [probability for _, probability in options]

    return random.choices(tokens, weights=probabilities, k=1)[0]


def generate_from_start(start, n=N, max_tokens=20, verbose=True):
    """Generate a continuation beginning with START."""
    generated = tokenize(start)

    if not generated:
        raise ValueError("START must contain at least one token.")

    if n < 2:
        raise ValueError("N must be at least 2.")

    # The model's state has N-1 tokens.
    # If START is shorter, prepend <s> markers.
    state_size = n - 1
    context = (["<s>"] * max(0, state_size - len(generated))
               + generated[-state_size:])

    for step in range(1, max_tokens + 1):
        history = tuple(context[-state_size:])
        next_token = sample_next(history)

        if verbose:
            print(
                f"Step {step:2d}: "
                f"state={history} -> next={next_token}"
            )

        if next_token is None:
            print("No transition was observed for this state.")
            break

        if next_token == "</s>":
            break

        generated.append(next_token)
        context.append(next_token)

    return " ".join(generated)


In [30]:
# ------------------------------------------------------------
# Generate one sentence
# ------------------------------------------------------------

print("Generated sentence:")
result = generate_from_start(
    START,
    n=N,
    max_tokens=MAX_GENERATED_TOKENS,
    verbose=True
)

print("\nFinal:", result)


Generated sentence:
Step  1: state=('s',) -> next=l
Step  2: state=('l',) -> next=r
Step  3: state=('r',) -> next=r
Step  4: state=('r',) -> next=r
Step  5: state=('r',) -> next=r
Step  6: state=('r',) -> next=r
Step  7: state=('r',) -> next=f
Step  8: state=('f',) -> next=e
Step  9: state=('e',) -> next=</s>

Final: s l r r r r r f e


## Try it again

If `RANDOM_SEED = None`, running the cell above again can produce a different continuation.

This is the key distinction between:

```text
probability model
```

and

```text
deterministic prediction
```

The model does not necessarily choose the most probable next token. It **samples** from the learned probability distribution.

For example, if the current state has:

```text
s -> f : 0.70
s -> p : 0.20
s -> e : 0.10
```

then repeated executions can produce different paths, while respecting those probabilities statistically.
